In [ ]:
# ─────────────────────────────────────────────────────
# BERTopic 분석 — 4개 브랜드 비교
# A. 전체 리뷰 → 브랜드별 주요 토픽 구조
# B. 저평점 리뷰 → 브랜드별 불만 토픽 드릴다운
# ─────────────────────────────────────────────────────

# ── 설치 ─────────────────────────────────────────────
# pip install bertopic sentence-transformers umap-learn hdbscan

In [1]:
import pandas as pd
import numpy as np
import os
from kiwipiepy import Kiwi
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

In [ ]:
# import torch
# print(torch.__version__)
# print(torch.__file__)          # .venv 경로여야 함
# print(torch.cuda.is_available())

In [34]:
# ── 사용자 사전 ───────────────────────────────────────────────
USER_DICT = {
    # 핏/사이즈
    "스탠다드핏",
    "오버핏",
    "슬림핏",
    "테이퍼드핏",
    "와이드핏",
    "레귤러핏",
    "머슬핏",
    "크롭핏",
    "루즈핏",
    "릴렉스핏",
    "한사이즈업",
    "한사이즈다운",
    "반사이즈업",
    "반사이즈다운",
    "정사이즈",
    "반업",
    "일업",
    "사이즈업",
    "허리",
    "골반",
    "허벅지",
    "종아리",
    "어깨",
    "밑위",
    "기장",
    "기장감",
    "암홀",
    "시보리",
    "하비",
    "하체비만",
    "힙딥",
    "승마살",
    "흉곽압박",
    "Y존",
    "와이존",
    "군살",
    "체형보정",
    "눈바디",
    "비율",
    "칼발",
    "발볼러",
    # 소재·내구성
    "기모",
    "약기모",
    "심리스",
    "쿨링",
    "메쉬",
    "안감",
    "신축성",
    "복원력",
    "비침",
    "땀흡수",
    "통기성",
    "보풀",
    "물빠짐",
    "이염",
    "변형",
    "내구성",
    "탄탄",
    "두께",
    "두께감",
    "터치감",
    "바스락",
    "텐셀",
    "모달",
    "나일론",
    "플리스",
    "코듀라",
    "스웻",
    "허리말림",
    "배말림",
    "세탁망",
    "건조기",
    # 착용감·기능성
    "착용감",
    "텐션감",
    "압박감",
    "밀착",
    "흡습속건",
    "활동성",
    "움직임",
    "미끄럼방지",
    "땀자국",
    # 스포츠 용도
    "크로스핏",
    "웨이트",
    "데드리프트",
    "스쿼트",
    "러닝",
    "하이킹",
    "요가",
    "필라테스",
    "웰니스",
    "리커버리",
    # 카테고리
    "러닝복",
    "요가복",
    "필라테스복",
    "골프웨어",
    "스윔웨어",
    "애슬레저",
    "레깅스",
    "팬츠",
    "조거팬츠",
    "슬랙스",
    "브라탑",
    "크롭티",
    "맨투맨",
    "쇼츠",
    "바람막이",
    # 디자인·스타일
    "고프코어",
    "발레코어",
    "올드머니룩",
    "Y2K",
    "꾸안꾸",
    "출근룩",
    "데일리룩",
    "원마일웨어",
    "셋업",
    "웜톤",
    "쿨톤",
    "톤다운",
    "무채색",
    "파스텔",
    "넥라인",
    "레이어드",
    "반집업",
    "핑거홀",
    # 브랜드 라인
    "에어쿨링",
    "에어리핏",
    "올데이핏",
    "에어무스",
    "에어스트",
    "에어캐치",
    "텐셀모달",
    "얼라인",
    "스쿠바",
    "원더트레인",
    "패스트앤프리",
    "디파인",
    "아시아핏",
    "눌루",
    "에버럭스",
    "블랙라벨",
    "아이스페더",
    "젤라",
    "에샤페",
    "인터런",
    "타르가",
    "디스럽터",
    "헤리티지",
    "어글리슈즈",
    "빅로고",
    # 브랜드 충성도
    "재구매",
    "교복템",
    "인생바지",
    "강추",
    "최애템",
    "실패없다",
    "후회없다",
    # 불만
    "반품",
    "교환",
    "환불",
    "불량",
    "실망",
    "불편",
    "아쉽다",
    "탈색",
    "늘어나다",
    "사이즈",
    "길이",
    "짧다",
    "길다",
    "타이트",
    "크다",
    "작다",
    # 가격
    "가성비",
    "가심비",
    "갓성비",
    "가격대비",
    "합리적",
    "할인",
    "특가",
    # 착용 불편
    "답답하다", "꽉끼다", "조이다", "눌리다", "걸리적",
    "흘러내리다", "말리다", "올라가다", "벌어지다",
    # 소재 불량
    "이염", "물빠짐", "탈색", "냄새", "식초", "올트임",
    "봉제선", "박음질", "하자", "불량", "구멍",
    # 착화감
    "아프다", "물집", "쓸리다", "발바닥",
    # CS 불만
    "환불불가", "교환거절", "검수불량", "재판매",
}

# ── 불용어 (c-TF-IDF 단계에서만 적용) ────────────────────────
# 원칙: 임베딩에는 원문 그대로, 키워드 추출 시에만 제거
STOP_WORDS = {
    "네이버",
    "페이",
    "후기",
    "작성",
    "등록",
    "포인트",
    "아울렛",
    "구매",
    "상품",
    "주문",
    "배송",
    "생각",
    "그냥",
    "진짜",
    "정도",
    "많이",
    "조금",
    "느낌",
    "착용",
    "구입",
    "종류",
    "보다",
    "있다",
    "없다",
    "같다",
    "되다",
    "하다",
    "이다",
    "받다",
    "사다",
    "입다",
    "좋다",
    "예쁘다",
    "편하다",
    "만족",
    "잘맞다",
    "괜찮다",
    "남편", "신랑", "아내", "아들", "딸", "아이", "엄마",
    "선물", "남자친구", "여자친구",
    "합리적", "감수", "희망", "지대", "직장",
    "화요일", "휴일", "각질", "소장", "타월",
    "아파트", "공원", "습관", "카톡", "커플",
    "대비", "가격",
    # 프로모션/할인 (구매 맥락이지 불만 속성 아님)
    "특가", "할인", "세일", "행사", "쿠폰",
    # 의미없는 평가어
    "보통",
    "평범",
    "무난",
    "그냥",
    "약간",
    "빠르다", "빠르", "포장", "도착",
    "조아요", "조아", "이뻐요", "넘", "넘나",
    # 추상적 감정/상황어 (불만 속성 설명 안 함)
    "각오", "민망하다", "피해자", "흔하다",
    "희망", "지대", "직장", "아파트", "공원",
    "습관", "카톡", "커플", "화요일", "휴일",
    "역대", "가관", "계산", "한마디", "항의",
    "폭발", "통보", "계기", "캡쳐", "연말",
    # 제품명이 키워드로 올라온 것 (너무 구체적)
    "피치라이닝", "윈드자켓", "골프 마스크",
    "후디", "프리런", "짐볼",
    # 기능 설명어 (불만 맥락 없이 중립적)
    "차단", "냉감",   # seed에서 제거도 검토
    # 브랜드명 (어느 토픽에나 공통 등장 → 변별력 없음)
    "안다르",
    "젝시믹스", "젝스믹스", "젝시미스", "젝스미스", 
    "룰루레몬",
    "휠라",
    "젝시",
    "젝믹",
    "룰루",
    # 색상
    "블랙", "화이트", "검정", "흰색", "아이보리", "베이지",
    "그레이", "네이비", "핑크", "카키", "브라운", "민트",
    "블루", "레드", "옐로우", "그린", "퍼플", "차콜",
    "색상", "컬러", "색깔", "색",
    # 제품명
    "페이스슬림", "라이트더블"
    
}

# ── Seed Topic List (소프트 가이드 — 군집 방향 유도) ─────────
# 주의: 클러스터 구조 자체를 바꾸지는 않음
#       UMAP+HDBSCAN 클러스터링 후 가장 유사한 토픽에 레이블 매핑
seed_topic_list = [
    # ── 핏/사이즈 (전 카테고리 저평점 1위) ──────────────────────
    [
        "하비",
        "골반",
        "Y존",
        "오버핏",
        "기장",
        "사이즈업",
        "한사이즈업",
        "밑위",
        "기장감",
        # EDA 추가
        "사이즈",
        "크다",
        "작다",
        "짧다",
        "길다",
        "타이트",
        "허리",
        "반업",
        "정사이즈",
        "한사이즈", "적다",
    ],
    # ── 소재·내구성 ────────────────────────────────────────────
    [
        "기모",
        "메쉬",
        "보풀",
        "이염",
        "변형",
        "탄탄",
        "물빠짐",
        "세탁망",
        "건조기",
        # EDA 추가
        "탈색",
        "내구성",
        "세탁",
        "세탁기",
        "얇다",
        "늘어나다",
        "헤지다",
        "빠짐",
        "심하다",
    ],
    # ── 기능성 ────────────────────────────────────────────────
    [
        "신축성",
        "땀흡수",
        "통기성",
        "흡습속건",
        "복원력",
        "미끄럼방지",
        "활동성",
        # EDA 추가
        "비침",
        "땀자국",
        "움직임",
    ],
    # ── 착용감·불편 ────────────────────────────────────────────
    # EDA 신규: 상의 타이트·불편, 하의 착용 시 불편
    [
        "착용감",
        "불편",
        "압박감",
        "조이다",
        "끼다",
        "허리말림",
        "배말림",
        "말리다",
        "올라가다",
        "흘러내리다",
        "텐션감",
        "밀착",
        "딱딱하다",
        "뻣뻣하다",
        "무겁다",
    ],
    # ── 신발·발 불편 ──────────────────────────────────────────
    # EDA 신규: 신발 카테고리 저평점 1~3위 — 아프다·발가락·발볼
    [
        "아프다",
        "발가락",
        "발볼",
        "발등",
        "발바닥",
        "꽉끼다",
        "좁다",
        "물집",
        "벗겨지다",
        "미끄럽다",
        "쿠션",
        "아치",
        "깔창",
        "착화감",
    ],
    # ── 언더웨어 불만 ─────────────────────────────────────────
    # EDA 신규: 세탁 후 자국·변형, 브라 끈 끊어짐
    [
        "자국",
        "땀자국",
        "브라",
        "팬티",
        "심리스",
        "일체",
        "후크",
        "끊어지다",
        "찢어지다",
        "올데이핏",
        "레깅스팬티",
        "브라탑",
    ],
    # ── CS·반품·검수 불만 ────────────────────────────────────
    [
        "반품",
        "교환",
        "환불",
        "불량",
        "보풀",
        "불편",
        "실망",
        # EDA 추가
        "검수",
        "고객",
        "전화",
        "연락",
        "센터",
        "처리",
        "포장",
        "비닐",
        "재판매",
    ],
    # ── 색상·디자인 불만 ─────────────────────────────────────
    # EDA 신규: 룰루레몬 저평점 — 화면과 실물 색상 차이
    [
        "색상",
        "화면",
        "실물",
        "다르다",
        "차이",
        "실망",
        "사진",
        "밝다",
        "어둡다",
        "톤",
        "컬러",
    ],
    # ── 디자인·무드 (긍정 탐색용) ───────────────────────────
    [
        "올드머니룩",
        "고프코어",
        "꾸안꾸",
        "웜톤",
        "발레코어",
        "Y2K",
        "셋업",
    ],
    # ── 브랜드 충성도·헤리티지 ────────────────────────────────
    [
        "재구매",
        "인생바지",
        "강추",
        "교복템",
        "후회없다",
        "실패없다",
        "최애템",
    ],
    # ── 가격·가성비 ───────────────────────────────────────────
    [
        "가성비",
        "가심비",
        "갓성비",
        "가격대비",
        "할인",
        # EDA 추가
        "특가",
        "비싸다",
        "아깝다",
    ],
]

print(f"✔ seed_topic_list: {len(seed_topic_list)}개 시드")
for i, seed in enumerate(seed_topic_list):
    print(f"  [{i:2d}] {seed[:3]} ...")

print(f"✔ USER_DICT: {len(USER_DICT)}개")
print(f"✔ STOP_WORDS: {len(STOP_WORDS)}개")
print(f"✔ seed_topic_list: {len(seed_topic_list)}개 시드")

✔ seed_topic_list: 11개 시드
  [ 0] ['하비', '골반', 'Y존'] ...
  [ 1] ['기모', '메쉬', '보풀'] ...
  [ 2] ['신축성', '땀흡수', '통기성'] ...
  [ 3] ['착용감', '불편', '압박감'] ...
  [ 4] ['아프다', '발가락', '발볼'] ...
  [ 5] ['자국', '땀자국', '브라'] ...
  [ 6] ['반품', '교환', '환불'] ...
  [ 7] ['색상', '화면', '실물'] ...
  [ 8] ['올드머니룩', '고프코어', '꾸안꾸'] ...
  [ 9] ['재구매', '인생바지', '강추'] ...
  [10] ['가성비', '가심비', '갓성비'] ...
✔ USER_DICT: 203개
✔ STOP_WORDS: 138개
✔ seed_topic_list: 11개 시드


In [35]:
# ─────────────────────────────────────────────────────────────
# Cell 2: KiwiPy + BERTopicVectorizer
# 핵심: vocabulary 고정 없이 tokenize_ko에서 USER_DICT 우선 포함
# ─────────────────────────────────────────────────────────────

_kiwi = Kiwi(num_workers=-1)
TARGET_TAGS = {"NNG", "NNP", "VA", "XR"}


def _tag_str(tag):
    return tag.name if hasattr(tag, "name") else str(tag)

# ── LEMMA_MAP (동의어 정규화) ─────────────────────────────────
LEMMA_MAP = {
    # 형태소 분석기 출력 정규화
    "편하":   "편하다",
    "이쁘":   "예쁘다",
    "귀여":   "귀엽다",
    # 동의어 통일
    "신축":   "신축성",  
    "데님":   "청바지",
    "진":     "청바지",
    "조거":   "조거팬츠",
    "레깅":   "레깅스",
    "스판":   "신축성",
    "스팬":   "신축성",
    "탄성":   "신축성",
    "향":     "냄새",
}

# ── NEG_STOPWORDS (저평점 전용 추가 불용어) ───────────────────
NEG_STOPWORDS = {
    "좋다", "편하다", "만족", "추천", "강추",
    "감사", "최고", "굿", "짱",
    "잘맞다", "딱맞다", "맞다",
    "부드럽다", "따뜻하다", "시원하다",
    "재구매", "또살다",
    "합리적",     # Topic 2, 13에서 키워드로 등장
    "인지도",     # 브랜드 긍정 표현
    "단정",       # 디자인 긍정
    "향수",       # 긍정 맥락
}

def tokenize_ko(text: str) -> list:
    """
    c-TF-IDF 단계 전용 토크나이저

    [핵심 설계]
    vocabulary 고정(CountVectorizer(vocabulary=...)) 대신
    tokenizer 단계에서 USER_DICT 단어를 우선 포함하는 방식 사용.

    이유:
    - vocabulary 고정 시: USER_DICT에 없는 단어는 전부 무시
      → 실제 토픽에서 중요한 단어(예: '냄새','찍찍이')가 누락될 수 있음
    - tokenizer 방식: USER_DICT 단어를 추가 포함하되 전체 어휘도 허용
      → 탐색적 분석에 더 적합, 예상치 못한 키워드 발견 가능

    구현:
    형태소 분석 결과 + USER_DICT 단어 우선 보장 (중복 없이)
    """
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        tokens = _kiwi.tokenize(text)
        result = []
        seen = set()  # ← 중복 제거용

        for t in tokens:
            if _tag_str(t.tag) in TARGET_TAGS and len(t.form) > 1:
                if t.form in STOP_WORDS:
                    continue
                word = LEMMA_MAP.get(t.form, t.form)
                # ── 중복 제거 ──────────────────────────────────
                if word in seen:
                    continue
                seen.add(word)
                result.append(word)

        # 2. USER_DICT 단어 우선 보장
        #    원문에 USER_DICT 단어가 포함되어 있으면 토큰 리스트 앞에 추가
        #    (형태소 분석기가 복합어를 쪼개서 누락되는 경우 방지)
        #    seen에 이미 있으면 추가 안 함 → 중복 방지
        user_dict_tokens = [
            w for w in USER_DICT
            if w in text and w not in seen
        ]
        # USER_DICT 토큰도 seen에 추가
        seen.update(user_dict_tokens)

        return user_dict_tokens + result
    except Exception:
        return []


def tokenize_ko_low(text: str) -> list:
    """저평점 전용 — NEG_STOPWORDS 추가 적용"""
    tokens = tokenize_ko(text)
    return [t for t in tokens if t not in NEG_STOPWORDS]


class BERTopicVectorizer(CountVectorizer):
    """
    BERTopic 내부 DataFrame 처리 버그 수정
    vocabulary 고정 없이 사용 → 전체 어휘 허용
    """

    def _extract(self, raw):
        if isinstance(raw, pd.DataFrame) and "Document" in raw.columns:
            return raw["Document"].tolist()
        if hasattr(raw, "tolist"):
            return raw.tolist()
        return list(raw)

    def fit(self, raw, y=None):
        return super().fit(self._extract(raw), y)

    def fit_transform(self, raw, y=None):
        return super().fit_transform(self._extract(raw), y)

    def transform(self, raw):
        return super().transform(self._extract(raw))


# 검증
test_cases = [
    "레깅스 보풀 사이즈 반품해요",  # USER_DICT: 보풀, 사이즈, 반품
    "신축성이 없어서 너무 불편해요",  # USER_DICT: 신축성, 불편
    "오버핏인데 기장이 길어서 교환했어요",  # USER_DICT: 오버핏, 기장, 교환
    "데님 청바지 스판이 없어서 불편해요",
    "신축성 좋고 편하고 최고예요",
]
print("=== tokenize_ko 검증 ===")
for t in test_cases:
    tokens = tokenize_ko(t)
    user_hits = [tok for tok in tokens if tok in USER_DICT]
    print(f"입력: {t[:30]}")
    print(f"  전체 토큰: {tokens[:8]}")
    print(f"  USER_DICT 매칭: {user_hits}\n")
    print(f"전체: {tokenize_ko(t)}")
    print(f"저평점: {tokenize_ko_low(t)}\n")

=== tokenize_ko 검증 ===
입력: 레깅스 보풀 사이즈 반품해요
  전체 토큰: ['보풀', '레깅스', '사이즈', '반품']
  USER_DICT 매칭: ['보풀', '레깅스', '사이즈', '반품']

전체: ['보풀', '레깅스', '사이즈', '반품']
저평점: ['보풀', '레깅스', '사이즈', '반품']

입력: 신축성이 없어서 너무 불편해요
  전체 토큰: ['신축성', '불편']
  USER_DICT 매칭: ['신축성', '불편']

전체: ['신축성', '불편']
저평점: ['신축성', '불편']

입력: 오버핏인데 기장이 길어서 교환했어요
  전체 토큰: ['오버핏', '기장', '교환']
  USER_DICT 매칭: ['오버핏', '기장', '교환']

전체: ['오버핏', '기장', '교환']
저평점: ['오버핏', '기장', '교환']

입력: 데님 청바지 스판이 없어서 불편해요
  전체 토큰: ['청바지', '신축성', '불편']
  USER_DICT 매칭: ['신축성', '불편']

전체: ['청바지', '신축성', '불편']
저평점: ['청바지', '신축성', '불편']

입력: 신축성 좋고 편하고 최고예요
  전체 토큰: ['신축성', '편하다', '최고']
  USER_DICT 매칭: ['신축성']

전체: ['신축성', '편하다', '최고']
저평점: ['신축성']



In [28]:
# ─────────────────────────────────────────────────────────────
# Cell 3: 데이터 로드 + 샘플링
# ─────────────────────────────────────────────────────────────

df_all = pd.read_parquet("./preprocessed_bertopic.parquet")
df_all = df_all[df_all["rating"] > 0].reset_index(drop=True)

print(f"전체: {len(df_all):,}건")
print(df_all["brand"].value_counts())

# 전체 리뷰: 브랜드별 최대 10만
SAMPLE_PER_BRAND = 100_000
df_sample = pd.concat(
    [group.sample(min(len(group), SAMPLE_PER_BRAND), random_state=42) for _, group in df_all.groupby("brand")]
).reset_index(drop=True)

# 저평점: 1~2점만 (3점 제외 — "보통" 토픽 방지)
df_low_bert = df_all[df_all["rating"] <= 2].reset_index(drop=True)

print(f"\n✔ 전체 샘플: {len(df_sample):,}건")
print(f"✔ 저평점(1~2점): {len(df_low_bert):,}건")
print(df_sample["brand"].value_counts())

전체: 1,140,506건
brand
안다르     634085
젝시믹스    475455
FILA     19569
룰루레몬     11397
Name: count, dtype: int64

✔ 전체 샘플: 230,966건
✔ 저평점(1~2점): 9,816건
brand
안다르     100000
젝시믹스    100000
FILA     19569
룰루레몬     11397
Name: count, dtype: int64


In [29]:
# Cell 4 임베딩 생성 전에 추가
import re

def clean_content(text):
    if not isinstance(text, str):
        return ""
    # 네이버 페이 타임스탬프 제거
    text = re.sub(
        r'\d{4}\s\d{2}\s\d{2}\s\d{2}\s\d{2}\s\d{2}\s에\s등록된\s네이버\s페이\s구매평',
        '', text
    )
    text = re.sub(r'네이버페이에서\s작성된\s후기입니다', '', text)
    return text.strip()

df_sample["content_clean"] = df_sample["content_clean"].apply(clean_content)
df_low_bert["content_clean"] = df_low_bert["content_clean"].apply(clean_content)

In [30]:
# ─────────────────────────────────────────────────────────────
# Cell 4: SBERT 임베딩 (캐싱)
# 원칙: 임베딩에는 원문 그대로 (불용어 제거 없음)
# ─────────────────────────────────────────────────────────────

embedding_model = SentenceTransformer("jhgan/ko-sroberta-multitask", device="cuda")

CACHE_FULL = "./data/embeddings_full.npy"
CACHE_LOW_12 = "./data/embeddings_low_1to2.npy"

# 전체 임베딩
if os.path.exists(CACHE_FULL):
    embeddings_full = np.load(CACHE_FULL)
    print(f"✔ 전체 임베딩 캐시 로드: {embeddings_full.shape}")
else:
    print("전체 임베딩 생성 중...")
    embeddings_full = embedding_model.encode(
        df_sample["content_clean"].fillna("").tolist(),  # 원문 그대로
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    np.save(CACHE_FULL, embeddings_full)
    print(f"✔ 저장 완료: {embeddings_full.shape}")

# 저평점 임베딩 (1~2점)
if os.path.exists(CACHE_LOW_12):
    embeddings_low = np.load(CACHE_LOW_12)
    print(f"✔ 저평점 임베딩 캐시 로드: {embeddings_low.shape}")
else:
    print("저평점 임베딩 생성 중...")
    embeddings_low = embedding_model.encode(
        df_low_bert["content_clean"].fillna("").tolist(),  # 원문 그대로
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    np.save(CACHE_LOW_12, embeddings_low)
    print(f"✔ 저장 완료: {embeddings_low.shape}")

# 빈 문서 제거 + 임베딩 동기화
valid_mask = df_low_bert["content_clean"].fillna("").str.strip() != ""
df_low_valid = df_low_bert[valid_mask].reset_index(drop=True)
embeddings_low_valid = embeddings_low[valid_mask.values]
print(f"✔ 저평점 유효: {len(df_low_valid):,}건 / {embeddings_low_valid.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✔ 전체 임베딩 캐시 로드: (230966, 768)
✔ 저평점 임베딩 캐시 로드: (9816, 768)
✔ 저평점 유효: 9,816건 / (9816, 768)


In [41]:
# ─────────────────────────────────────────────────────────────
# Cell 5: A. 전체 리뷰 BERTopic
# ─────────────────────────────────────────────────────────────

print("=== A. 전체 리뷰 BERTopic ===")

# 임베딩은 원문으로 생성했으므로 docs도 원문 사용
docs_full = df_sample["content_clean"].fillna("").tolist()

topic_model = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    umap_model=UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
    ),
    hdbscan_model=HDBSCAN(
        min_cluster_size=50,
        min_samples=10,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    ),
    # vocabulary 고정 없이 — tokenize_ko가 USER_DICT 우선 포함
    vectorizer_model=BERTopicVectorizer(
        tokenizer=tokenize_ko,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.9,
    ),
    ctfidf_model=ClassTfidfTransformer(
        reduce_frequent_words=True,
    ),
    seed_topic_list=seed_topic_list,  # 소프트 가이드
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)

topics_full, probs_full = topic_model.fit_transform(
    docs_full,
    embeddings=embeddings_full,
)
df_sample["topic"] = topics_full

outlier_ratio = (pd.Series(topics_full) == -1).mean()
print(f"\n초기 토픽 수: {len(topic_model.get_topic_info()) - 1}개")
print(f"Outlier 비율: {outlier_ratio:.1%}")

if outlier_ratio > 0.40:
    print("→ reduce_outliers 실행 중...")
    new_topics_full = topic_model.reduce_outliers(
        docs_full,
        topics_full,
        probabilities=probs_full,
        strategy="probabilities",
    )
    topic_model.update_topics(
        docs_full,
        topics=new_topics_full,
        vectorizer_model=topic_model.vectorizer_model,
    )
    df_sample["topic"] = new_topics_full
    print(f"Outlier 비율 (재배정 후): {(pd.Series(new_topics_full) == -1).mean():.1%}")

print("\n상위 15개 토픽:")
print(topic_model.get_topic_info().head(16).to_string(index=False))

=== A. 전체 리뷰 BERTopic ===


2026-05-06 14:01:53,490 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-06 14:01:55,562 - BERTopic - Guided - Completed ✓
2026-05-06 14:01:55,564 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 14:08:28,019 - BERTopic - Dimensionality - Completed ✓
2026-05-06 14:08:28,029 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 14:26:44,401 - BERTopic - Cluster - Completed ✓
2026-05-06 14:26:44,534 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 14:30:30,460 - BERTopic - Representation - Completed ✓



초기 토픽 수: 332개
Outlier 비율: 46.8%
→ reduce_outliers 실행 중...


2026-05-06 14:31:26,936 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outlier 비율 (재배정 후): 0.0%

상위 15개 토픽:
 Topic  Count                     Name                                                        Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [42]:
# ─────────────────────────────────────────────────────────────
# Cell 6: B. 저평점 BERTopic (1~2점)
# ─────────────────────────────────────────────────────────────

print("=== B. 저평점 BERTopic (1~2점) ===")

docs_low = df_low_valid["content_clean"].fillna("").tolist()

topic_model_low = BERTopic(
    embedding_model=embedding_model,
    language="multilingual",
    umap_model=UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
    ),
    hdbscan_model=HDBSCAN(
        min_cluster_size=40,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    ),
    vectorizer_model=BERTopicVectorizer(
        tokenizer=tokenize_ko_low,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
    ),
    ctfidf_model=ClassTfidfTransformer(
        reduce_frequent_words=True,
    ),
    # 저평점용 시드: 불만 속성 중심 (EDA 발굴 키워드 반영)
    seed_topic_list=[
    # 핏/사이즈
    [
        "사이즈", "크다", "작다", "기장", "짧다", "길다", "길이",
        "한사이즈업", "타이트", "허리", "골반", "밑위",
        "반업", "정사이즈", "스몰", "소매", "한사이즈",
        # 추가: EDA에서 확인된 불만 표현
        "봉제선", "박음질", "삐뚤다", "비대칭",
        # 제거: "기장감" → STOP_WORDS로 이동 (노이즈 반복)
    ],

    # 소재·내구성
    [
        "보풀", "이염", "물빠짐", "변형", "내구성", "탈색",
        "세탁", "건조", "얇다", "늘어나다", "먼지",
        # 추가
        "냄새", "식초", "올트임", "구멍", "뜯어지다", "해지다",
        # 제거: "헤지다", "심하다" → 너무 범용적
    ],

    # 착용감·불편
    [
        "불편", "압박감", "조이다", "끼다",
        "허리말림", "배말림", "말리다", "올라가다", "흘러내리다", "내려가다",
        "밀착", "딱딱하다", "무겁다", "뻣뻣하다",
        # 추가
        "답답하다", "눌리다", "걸리적", "쓸리다", "당기다",
        # 제거: "착용감", "텐션감" → 긍정에도 쓰여서 노이즈 유발
    ],

    # 신발·발 불편
    [
        "아프다", "발가락", "발볼", "발등", "발바닥",
        "꽉끼다", "좁다", "물집", "벗겨지다",
        "쿠션", "아치", "깔창", "착화감",
        # 추가
        "미끄럽다", "미끄러지다", "발목",
        # 제거: "미끄럼방지" → 긍정 키워드
    ],

    # 언더웨어 불만 → 소재 시드에 통합 (키워드 너무 적어 독립 시드 효과 없음)
    # 제거

    # CS·반품·검수
    [
        "반품", "교환", "환불", "불가", "불량", "검수", "불만", "불쾌감",
        "고객", "전화", "연락", "센터", "처리", "포장", "비닐", "재판매",
        # 추가: 실제 리뷰에서 반복 확인된 표현
        "누락", "오배송", "지연", "묵묵부답", "소비자기만",
    ],

    # 색상·디자인 불만
    [
        "화면", "실물", "다르다", "차이", "사진",
        "밝다", "어둡다",
        # 제거: "색상", "컬러", "톤", "디자인" → 너무 범용적, 긍정에도 등장
        # 제거: "촌스럽다", "두꺼워보이다", "어색하다" → 너무 희귀
    ],

    # 가격·가성비
    [
        "비싸다", "아깝다", "돈아깝다", "가격",
        # 제거: "특가" → STOP_WORDS로 이동
    ],
],
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)

topics_low, probs_low = topic_model_low.fit_transform(
    docs_low,
    embeddings=embeddings_low_valid,
)
df_low_valid["topic"] = topics_low

outlier_low = (pd.Series(topics_low) == -1).mean()
print(f"\n초기 토픽 수: {len(topic_model_low.get_topic_info()) - 1}개")
print(f"Outlier 비율: {outlier_low:.1%}")

if outlier_low > 0.30:
    print("→ reduce_outliers 실행 중...")
    new_topics_low = topic_model_low.reduce_outliers(
        docs_low,
        topics_low,
        probabilities=probs_low,
        strategy="probabilities",
    )
    topic_model_low.update_topics(
        docs_low,
        topics=new_topics_low,
        vectorizer_model=topic_model_low.vectorizer_model,
    )
    df_low_valid["topic"] = new_topics_low
    print(f"Outlier 비율 (재배정 후): {(pd.Series(new_topics_low) == -1).mean():.1%}")

print(f"\n불만 토픽 수: {len(topic_model_low.get_topic_info()) - 1}개")
print(topic_model_low.get_topic_info().head(16).to_string(index=False))

=== B. 저평점 BERTopic (1~2점) ===


2026-05-06 14:45:16,412 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-06 14:45:16,980 - BERTopic - Guided - Completed ✓
2026-05-06 14:45:16,980 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-06 14:45:37,884 - BERTopic - Dimensionality - Completed ✓
2026-05-06 14:45:37,884 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-06 14:45:38,644 - BERTopic - Cluster - Completed ✓
2026-05-06 14:45:38,649 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-06 14:45:56,731 - BERTopic - Representation - Completed ✓



초기 토픽 수: 11개
Outlier 비율: 0.2%

불만 토픽 수: 11개
 Topic  Count                Name                                    Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [43]:
# ─────────────────────────────────────────────────────────────
# Cell 7: 품질 검증
# ─────────────────────────────────────────────────────────────


def user_dict_coverage(model, user_dict, label="", top_n=10):
    """USER_DICT 단어가 토픽 키워드로 얼마나 올라왔는지 검증"""
    total, matched = 0, 0
    for tid in model.get_topic_info()["Topic"]:
        if tid == -1:
            continue
        kws = [w for w, _ in model.get_topic(tid)[:top_n]]
        total += len(kws)
        matched += sum(1 for w in kws if w in user_dict)
    rate = matched / total * 100 if total > 0 else 0
    print(f"[{label}] USER_DICT 커버리지: {matched}/{total} ({rate:.1f}%)")
    return rate


def show_topic_examples(model, docs, topics, top_k=10, per_topic=3):
    """토픽 키워드 + 대표 리뷰 확인 (반드시 실행)"""
    info = model.get_topic_info()
    topic_ids = info.loc[info["Topic"] != -1, "Topic"].head(top_k).tolist()
    for tid in topic_ids:
        kws = [w for w, _ in model.get_topic(tid)[:10]]
        idxs = [i for i, t in enumerate(topics) if t == tid][:per_topic]
        print(f"\n===== Topic {tid} =====")
        print(f"키워드: {kws}")
        print("대표 리뷰:")
        for i in idxs:
            print(f"  - {docs[i][:100]}")


# USER_DICT 커버리지 확인
print("=== 품질 검증 ===")
user_dict_coverage(topic_model, USER_DICT, label="전체")
user_dict_coverage(topic_model_low, USER_DICT, label="저평점")

# 커버리지 60% 미만이면 경고
# → USER_DICT 확장 또는 tokenize_ko 튜닝 필요

print("\n=== 저평점 대표 리뷰 확인 ===")
show_topic_examples(
    topic_model_low,
    docs_low,
    df_low_valid["topic"].tolist(),
    top_k=10,
)

=== 품질 검증 ===
[전체] USER_DICT 커버리지: 36/3320 (1.1%)
[저평점] USER_DICT 커버리지: 45/110 (40.9%)

=== 저평점 대표 리뷰 확인 ===

===== Topic 0 =====
키워드: ['실물', '소매', '불만', '길다', '물빠짐', '걸리적', '스몰', '변형', '작다', '발가락']
대표 리뷰:
  - 막상 구매하니 손이 잘 안가네요 릴렉스 마사지 듀얼볼 ⅱ 후기
  - 불편해요 요가수업 끝나고 얼른 정리해거 나가고 싶은데 스트랩이 정리를 편하게 도와주지는 않네요 그냥 요가매트만 둘둘 말아서 나가는게 더 편해요ㅠㅠ 단 색상은 이쁩니다  불만족
  - 예뻐요 요가매트 스트랩 위드 후기 예뻐요

===== Topic 1 =====
키워드: ['실물', '작다', '물빠짐', '스몰', '소매', '변형', '불만', '이염', '아프다', '화면']
대표 리뷰:
  - 먼지가 많이 붙는 재질인게 좀 아쉽네요  먼지가 많이 붙는 재질인게 좀 아쉽네요
  - 블랙은 너무 더러워지는게 잘 보이네요  불만족
  - 한달동안 사용하시면서 달라진  불만족

===== Topic 2 =====
키워드: ['길다', '불만', '크다', '비닐', '식초', '불가', '화면', '전화', '쿠션', '발목']
대표 리뷰:
  - 반품은 귀찮으니 핀 보내주시길요 답변부탁요 릴렉스 짐볼 65cm 후기 빠른 처리 부탁요
  - 누가봐도 반품 상품인거 보냄 씰링 뜯어진 상태로 박스에 붙어있고 반품된거 검수는 안했는지 바람 마개가 안왔음ㅋㅋ 중고 쓴다 생각하고 10분 넘게 바람 넣었더니 마개가 없다ㅋ 릴렉스
  - 국내 배송이 일주일 정도 걸리는 시스템이 이해가 안됩니다 취소한려다 그냥 씁니다  국내 배송이 일주일 정도 걸리는 시스템이 이해가 안됩니다 취소한려다 그냥 씁니다

===== Topic 3 =====
키워드: ['작다', '건조', '아프다', '크다', '불가', '비닐', '발등', '발가락', '발

In [44]:
# ─────────────────────────────────────────────────────────────
# Cell 8: 브랜드별 토픽 분포
# ─────────────────────────────────────────────────────────────


def get_topic_name(model, topic_id, top_n=3):
    kws = model.get_topic(topic_id)
    if not kws or not kws[0][0]:
        return f"토픽_{topic_id}"
    return "_".join([w for w, _ in kws[:top_n] if w])


for label, df_target, model in [
    ("전체", df_sample, topic_model),
    ("저평점", df_low_valid, topic_model_low),
]:
    df_target["topic_name"] = df_target["topic"].apply(lambda t: get_topic_name(model, t) if t != -1 else "노이즈")
    brand_topic = df_target[df_target["topic"] != -1].groupby(["brand", "topic_name"]).size().reset_index(name="count")
    brand_topic["ratio"] = brand_topic.groupby("brand")["count"].transform(lambda x: x / x.sum() * 100).round(1)

    print(f"\n=== 브랜드별 상위 5 토픽 ({label}) ===")
    for brand in df_target["brand"].unique():
        top5 = brand_topic[brand_topic["brand"] == brand].sort_values("ratio", ascending=False).head(5)
        print(f"\n[{brand}]")
        for _, row in top5.iterrows():
            print(f"  {row['topic_name']:<35} {row['ratio']:.1f}%")


=== 브랜드별 상위 5 토픽 (전체) ===

[FILA]
  발볼_갓성비_확률                           6.9%
  퓨어그레이_크다 데일리룩_카멜색                   3.8%
  발볼_디스커버리_테니스장                       3.7%
  확률_포스_경주                            3.7%
  갓성비_야간 안전_다른컬로                      3.7%

[룰루레몬]
  갓성비_메일_가슴 둘레                        8.7%
  흡습속건_해녀_발수                          4.2%
  페일핑크_프로필 촬영_미음                      3.4%
  필라테스복 할인_펄렁_죠아용                     3.1%
  발수_합리적 출근룩_감정                       2.6%

[안다르]
  흡습속건_갓성비_실버그레이                      6.4%
  세탁망 한사이즈다운_히프_힌치수                   4.6%
  가뱝고_시어서커 하이_스키니핏                    3.1%
  프라임니트 하이넥_겨울 제외_박시한                 2.5%
  체계_요상하_장말                           2.4%

[젝시믹스]
  흡습속건_해녀_발수                          2.9%
  간과_청록_로고 플레이                        2.9%
  효용_토트백_검덩                           2.9%
  바디 프로필_강림_바위                        2.7%
  부서_가뱝고_오이스터                         2.4%

=== 브랜드별 상위 5 토픽 (저평점) ===

[안다르]
  실물_소매_불만                            78.9%
  실

In [45]:
# ─────────────────────────────────────────────────────────────
# Cell 9: 저장
# ─────────────────────────────────────────────────────────────

import pickle

os.makedirs("./models", exist_ok=True)

# tokenize_ko는 명시적 함수라 pickle 문제 없음
with open("./models/bertopic_full.pkl", "wb") as f:
    pickle.dump(topic_model, f)
with open("./models/bertopic_low.pkl", "wb") as f:
    pickle.dump(topic_model_low, f)

df_sample.to_csv("./data/bertopic_full_result.csv", index=False, encoding="utf-8-sig")
df_low_valid.to_csv("./data/bertopic_low_result.csv", index=False, encoding="utf-8-sig")

print("✔ 저장 완료")
print(f"  전체 토픽: {len(topic_model.get_topic_info())-1}개")
print(f"  저평점 토픽: {len(topic_model_low.get_topic_info())-1}개")

✔ 저장 완료
  전체 토픽: 331개
  저평점 토픽: 11개
